K-fold Кросс-валидация - метод оценки обобщающей способности модели, при котором обучающую выборку несколько разбивают на обучающую и валидационную части(такие части называются фолдами), каждый раз обучают модель на одной части и валидируют на другой, после чего полученные результаты усредняются  
Например, при 4-fold cross validation датасет разбивают на 4 части, из которых 3 используют для обучения, а один - для валидации. Так происходит 4 раза, чтобы каждый фолд побывал в валидационной части. В итоге мы получаем 4 метрики, которые усредняем  

Также есть Leave One Out Cross Validation. Это такая кросс-валидация, при котором фолдом является объект в датасете, то есть фолдов столько же, сколько объектов. Механика такая же, как описано выше

Вообще, я использовал кросс-валидацию, когда писал stacking в предыдущем ноутбуке, но мне нужно понимание. Плюсом, я очень искусственно заставил количество фолдов зависить от количество переданных моделей в ансамбль, не знаю почему

Реализуем кросс-валидацию, сравнивать будем логистическую регрессию, kNN, решающее дерево и случайный лес. Градиентный бустинг дисквалифицирован по причине кака


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_covtype
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

In [2]:
data = fetch_covtype()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Target'] = data.target
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 581012 entries, 0 to 581011
Data columns (total 55 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Elevation                           581012 non-null  float64
 1   Aspect                              581012 non-null  float64
 2   Slope                               581012 non-null  float64
 3   Horizontal_Distance_To_Hydrology    581012 non-null  float64
 4   Vertical_Distance_To_Hydrology      581012 non-null  float64
 5   Horizontal_Distance_To_Roadways     581012 non-null  float64
 6   Hillshade_9am                       581012 non-null  float64
 7   Hillshade_Noon                      581012 non-null  float64
 8   Hillshade_3pm                       581012 non-null  float64
 9   Horizontal_Distance_To_Fire_Points  581012 non-null  float64
 10  Wilderness_Area_0                   581012 non-null  float64
 11  Wilderness_Area_1                   5

In [3]:
X = df.drop(columns=['Target'])
y = df['Target']

In [4]:
class KFoldCrossValidation:
    def crossValidation(self, model, X, y, k):#очев же, что X и y это дата, а k - количество фолдов.
        #Кросс валидация имеет смысл только при 2<=k<=len(X)
        fold_size = len(X)// k
        remainder = len(X)%k
        folds = [fold_size]*k
        score = 0
        for i in range(remainder):
            folds[i] = fold_size + 1
        start = 0
        for fold_size in folds:#а я ведь мог бы использовать sklearn.model_selection.KFold(), а не заниматься такой фигнёй...
            X_fold = X[start : start+fold_size]
            y_fold = y[start : start+fold_size] 
            X_out_fold = np.concatenate([X[:start], X[start+fold_size:]])
            y_out_fold = np.concatenate([y[:start], y[start+fold_size:]])
            #допустим, моя модель требует скейлера. НЕЛЬЗЯ СНАЧАЛА ПРОСКЕЙЛИТЬ ВСЮ ДАТУ, А ПОТОМ KFold, получим data leakage. Делаем так
            scaler = StandardScaler()
            X_out_fold = scaler.fit_transform(X_out_fold)
            X_fold = scaler.transform(X_fold)
            start +=fold_size
            model.fit(X_out_fold, y_out_fold)
            y_pred = model.predict(X_fold)
            score+=f1_score(y_fold, y_pred, average='macro')
        return score/k


Пора сравнивать наверное


In [5]:
kf = KFoldCrossValidation()
model = LogisticRegression(max_iter=1000)
metric = kf.crossValidation(model, X, y, 10)#без train_test_split, потому что мы всё разобьём в кросс валидации
print(metric)

C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.w

0.4389580271581437


In [6]:
kf = KFoldCrossValidation()
model = DecisionTreeClassifier()
metric = kf.crossValidation(model, X, y, 10)
print(metric)

C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.w

0.5297334989473487


In [7]:
kf = KFoldCrossValidation()
model = RandomForestClassifier()
metric = kf.crossValidation(model, X, y, 10)
print(metric)

C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.w

0.6411386449851978


In [8]:

kf = KFoldCrossValidation()
model = KNeighborsClassifier()
metric = kf.crossValidation(model, X, y, 10)
print(metric)

C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\hira0mi\Desktop\job_preparation\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.w

0.5256599338110141


| Model                      | f1 score |
| -------------------------- | ---: |
| LogisticRegression          |  0.44 |
| Decision Tree              |  0.53 |
| Random Forest                 |  0.64 |
| kNN                |  0.53 |

Ну само собой можно было не особо париться и сделать по коду ну куда короче, используя sklearn.pipeline.Pipeline и sklearn.model_selection.cross_val_score
sklearn.pipeline.Pipeline позволяет не париться о последовательности действий(как я парился по поводу скейлера и кросс-валидации), склёрн всё сделает сам и без data leakage

In [15]:


from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
X = df.drop(columns=['Target'])
y = df['Target']
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])
scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring=make_scorer(f1_score, average='macro')
)
print(scores)

[0.52982956 0.53609643 0.49492906 0.51736813 0.60231013]


тут прям инфа по каждому фолду. Взять это, просуммировать и разделить на количество фолдов не составить проблем

In [16]:
print(sum(scores)/len(scores))

0.5361066607523758


даже чуть выше стало. Ну крута же